# Beijing PM2.5 Forecasting  
## Notebook 05 — Time-Based Train / Validation / Test Split

In this notebook, we split the feature-engineered dataset into training, validation, and test sets using **time-based splitting**.

Random splitting is **not appropriate** for time-series forecasting, as it would cause data leakage from the future into the past.

In [16]:
import pandas as pd

## 1. Load Feature-Engineered Dataset

We load the dataset created in `04_feature_engineering.ipynb`. This dataset contains only valid, fully engineered rows.

In [17]:
DATA_PATH = "../data/processed/aotizhongxin_features.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"], index_col="datetime")

df.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,...,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
datetime,,,,,,,,,,,,,,,,,,,,,
2013-03-02 00:00:00,22.0,24.0,24.0,44.0,500.0,44.0,-0.4,1031.0,-17.6,0.0,...,False,False,False,False,False,False,False,False,False,False
2013-03-02 01:00:00,14.0,17.0,21.0,36.0,400.0,50.0,-1.0,1031.3,-17.3,0.0,...,False,False,False,False,False,False,False,False,False,False
2013-03-02 02:00:00,13.0,13.0,20.0,37.0,400.0,47.0,-1.5,1030.9,-16.9,0.0,...,False,False,False,False,False,False,False,False,False,False
2013-03-02 03:00:00,3.0,9.0,13.0,34.0,400.0,52.0,-1.4,1030.6,-17.6,0.0,...,False,False,False,False,False,False,False,False,False,False
2013-03-02 04:00:00,3.0,7.0,18.0,43.0,400.0,43.0,-1.5,1030.8,-17.7,0.0,...,True,False,False,False,False,False,False,False,False,False


## 2. Define Target and Feature Matrix

We predict PM2.5 at time *t* using all other features.

In [18]:
# Seperate x and y
TARGET = "PM2.5"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (33878, 45)
Target vector shape: (33878,)


## 3. Why Time-Based Splitting

In time-series forecasting:
- Future data must never influence past predictions
- Random shuffling introduces leakage
- Models must be evaluated on unseen future periods

We therefore split the dataset **chronologically**.

## 4. Split Strategy

We split the data as follows:

- **70% Training** — model learning
- **15% Validation** — hyperparameter tuning
- **15% Test** — final, unbiased evaluation

All splits follow the natural time order of the data.

In [19]:
# Perform time based split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]


In [20]:
# Verify split sizes
print("Train set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Train set: (23714, 45) (23714,)
Validation set: (5082, 45) (5082,)
Test set: (5082, 45) (5082,)


## 5. Temporal Boundary Check

We confirm that:
- training data ends before validation
- validation ends before test
- no overlap exists

In [21]:
# Time range check
print("Train period:", X_train.index.min(), "→", X_train.index.max())
print("Validation period:", X_val.index.min(), "→", X_val.index.max())
print("Test period:", X_test.index.min(), "→", X_test.index.max())

Train period: 2013-03-02 00:00:00 → 2015-12-31 07:00:00
Validation period: 2015-12-31 08:00:00 → 2016-08-01 02:00:00
Test period: 2016-08-01 03:00:00 → 2017-02-28 23:00:00


## 6. Save Split Datasets

These files will be used directly for model training.

In [22]:
X_train.to_csv("../data/processed/X_train.csv")
y_train.to_csv("../data/processed/y_train.csv")

X_val.to_csv("../data/processed/X_val.csv")
y_val.to_csv("../data/processed/y_val.csv")

X_test.to_csv("../data/processed/X_test.csv")
y_test.to_csv("../data/processed/y_test.csv")

print("Train / validation / test sets saved successfully.")

Train / validation / test sets saved successfully.


## Train-Test Split Summary

- The dataset was split chronologically to prevent data leakage
- Validation and test sets represent unseen future periods
- This setup reflects real-world forecasting conditions

The data is now ready for baseline and advanced model training.